# Stats 102B Homework 2

## **Author:** Bosen Yang

# PROBLEM 1
The **negative log-likelihood function** is given as:
$$\ell(\beta)=-\frac{1}{m}\sum_{i=1}^m [y_i \log(\pi(x_i,\beta))+(1-y_i) \log(1-\pi(x_i,\beta))]$$
where $y_i \in \{0,1\}$, $x_i \in \mathbb{R}^p$, and $\beta \in \mathbb{R}^p$. The probability function is defined as
$$\pi(x_i,\beta)=\frac{1}{1+\exp(-x_i^T\beta)}$$

To obtain estimates of the regression coefficients $\beta$, we first compute the gradient of $\ell(\beta)$:

$$\nabla \ell(\beta)=\frac{1}{m}\sum_{i=1}^m (\nabla_{\beta}[-y_i (x_i^T\beta)] + \nabla_{\beta}[\log(1+\exp(x_i^T\beta))])=\frac{1}{m}\sum_{i=1}^m (-y_ix_i+\sigma(x_i^T\beta)\cdot x_i)$$

where $\sigma(z) = \frac{1}{1+\exp(-z)}$ is the logistic sigmoid function.


## Part (a): Hyperparameter Tuning

### 1) For SGD with Nesterov momentum: How did you select the step size $\eta_k$ and the momentum factor?

**Step size:** I select the initial step size empirically by trying different values in descending order to ensure convergence rate. The choice is based on the which one gives the best prediction accuracy.

**Momentum parameter:** I don't make the momentum parameter to be fixed; instead, I used an increasing schedule $\xi_k = \frac{k-1}{k+2}$ to make the momentum grows over iterations and approaches to 1, which helps accelerate convergence.

### 2) For SGD with decreasing step sizes: What specific schedule did you select?

I make the step size slowly decay by applying $\eta_k = \frac{\eta_0}{\sqrt{k}}$. This makes sure step size does not dramatically drop in few iterations but also does not make it strickly 0.

### 3) For AMSGrad Adam: How did you select the $\beta_1$ and $\beta_2$ parameters, as well as the initial step size $\eta_0$

I adopted standard values $\beta_1 = 0.9$ and $\beta_2 = 0.99$ shown in the lecture slides. Those values are widely adopted in practice for a good balance between stability and adaptivity. Similarly, the initial step size $\eta_0$ was selected empirically by trying from large values and gradually shrink it. With a fixed number of iteration, the choice will be whichever gives the best prediction score.

### 4) For AMSGrad AdamW: How did you select $\beta_1$, $\beta_2$, $\eta_0$, and the weight decay parameter $\lambda$?

For momentum coefficients, applied the same values which are $\beta_1 = 0.9$ and $\beta_2 = 0.99$. Similarly, $\eta_0$ was selected by empirically trying values in descending orders. The weight decay parameter $\lambda$ also tuned empirically, and I choose a very small value that works better with generalization without overly shrinking the coefficients.

## Part (b): Part (b): Experimental Comparisons

### For all four algorithms, conduct experiments to evaluate the effect of the following mini-batch sizes (s): s $\in$ {1,10,50,500}.

The estimation error is $\|\hat{\beta}_{algo}-\hat{\beta}_{GLM}\|^2_2$

#### 3-5 Estimation Error

| Method    | s=1            | s=10           | s=50           | s=500          |
|-----------|---------------:|---------------:|---------------:|---------------:|
| Nesterov  | 34947010384.13 | 34954912393.39 | 34944359658.92 | 34944961959.19 |
| SGD       | 34945028469.16 | 34945028515.26 | 34945028395.97 | 34945028397.93 |
| Adam      | 34945105565.63 | 34945200996.31 | 34945877314.61 | 34947620754.58 |
| AdamW     | 34945226034.27 | 34945308452.26 | 34946709521.34 | 34948822956.10 |

#### 3-5 Iterations

| Method    | s=1  | s=10 | s=50 | s=500 |
|-----------|-----:|-----:|-----:|------:|
| Nesterov  | 1580 | 9700 | 6200 | 1580  |
| SGD       | 3320 | 2140 | 5260 | 6860  |
| Adam      | 3460 | 1680 | 5120 | 6320  |
| AdamW     | 2520 | 1820 | 4320 | 5520  |


#### 4-9 Estimation Error

| Method    | s=1           | s=10          | s=50          | s=500         |
|-----------|--------------:|--------------:|--------------:|--------------:|
| Nesterov  |48576513197.90 |48576224165.16 |48575965866.84 |48576083376.01 |
| SGD       |48576107544.58 |48576107605.49 |48576107593.29 |48576107595.55 |
| Adam      |48576108706.09 |48575963104.60 |48575664052.80 |48575512149.17 |
| AdamW     |48576108042.24 |48576083678.36 |48575468085.25 |48575637634.06 |

#### 4-9 Iterations

| Method    | s=1  | s=10 | s=50 | s=500 |
|-----------|-----:|-----:|-----:|------:|
| Nesterov  | 4840 | 4740 | 5360 | 1520  |
| SGD       | 2760 | 800  | 680  | 540   |
| Adam      | 1160 | 1720 | 7060 | 5440  |
| AdamW     | 1700 | 2220 | 1440 | 1080  |


**Observations:**  
- AdamW and Adam generally gives the smallest error, and Nesterov gives the largest error.
- AdamW and Adam usually takes much smaller step size to converge faster.
- In general, as batch-size increases, algorithms especially AdamW and Adam decreases its iteration.


# Appendix (Code)

In [5]:
#### --- IMPORTS --- ####
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [7]:
#### --- DEFINE FUNCTIONS --- ####

# Euclidean norm
def norm2(v):
    v = np.asarray(v)
    return np.sqrt(np.sum(v ** 2))

# Logistic Sigmoid function
def sigmoid(z):
    z = np.asarray(z)
    z = np.clip(z, -100, 100)
    return 1 / (1 + np.exp(-z))

# Objective function (to minimize)
def fcn(X, y, beta):
    m = X.shape[0]
    return - (1/m) * np.sum(y * np.log(sigmoid(X @ beta) + 1e-8) 
                            + (1 - y) * np.log(1 - sigmoid(X @ beta) + 1e-8))

# Gradient
def gradient(X, y, beta):
    m = X.shape[0]
    return (1/m) * (X.T @ (sigmoid(X @ beta) - y)) 

## 1. Stochastic Gradient Descent (SGD) with Nesterov momentum.

In [9]:
def Nesterov_SGD(gradient, X, y, batch_size = 25, step_size = 0.05):
    # Parameters
    tol = 1e-4
    tot_iter = 100000

    # Setup
    num_obs = X.shape[0]
    beta = np.zeros(X.shape[1])
    last_beta = beta.copy()
    prev_loss = fcn(X, y, beta)
    num_iter = 0
    
    for k in range(1, tot_iter + 1):
        # Increment
        num_iter += 1
        
        # Update Momentum
        xi = (k - 1) / (k + 2)
        
        # EXTRAPOLATION
        extrapolate = beta + xi * (beta - last_beta)

        # SUBSETING OBS RANDOMLY (SGD)
        index = np.random.choice(num_obs, batch_size, replace=False)
        X_sub = X[index]
        y_sub = y[index]

        # COMPUTE GRADIENT
        grad = gradient(X_sub, y_sub, extrapolate)
        
        # COMPUTE NEW POINT
        new_beta = extrapolate - step_size * grad
        last_beta = beta
        beta = new_beta

        # STOP CRITERION (CHECK EVERY 20 TIMES FOR STABILITY)
        if k % 20 == 0:
            curr_loss = fcn(X, y, beta)
            relative = abs(curr_loss - prev_loss) / (abs(prev_loss) + 1e-8)

            if relative < tol:
                print(f"Nesterov Converged at iteration {k}")
                break
                
            prev_loss = curr_loss

    return {"value": beta, "iterations": num_iter}

## 2. Stochastic Gradient Descent with a fixed schedule of decreasing step sizes.

In [11]:
def SGD(gradient, X, y, batch_size = 25, step_size_0 = 0.1):
    # Parameters
    tol = 1e-4
    tot_iter = 100000
    
    # Setup
    num_obs = X.shape[0]
    beta = np.zeros(X.shape[1])
    prev_loss = fcn(X, y, beta)
    num_iter = 0
    
    for k in range(1, tot_iter + 1):
        # Increment
        num_iter += 1
        
        # STEP SIZE DECAY
        step_size = step_size_0 / np.sqrt(k)

        # SUBSETING OBS RANDOMLY (SGD)
        index = np.random.choice(num_obs, batch_size, replace=False)
        X_sub = X[index]
        y_sub = y[index]

        # COMPUTE GRADIENT
        grad = gradient(X_sub, y_sub, beta)
        
        # COMPUTE NEW POINT
        beta = beta - step_size * grad

        # STOP CRITERION (CHECK EVERY 20 TIMES FOR STABILITY)
        if k % 20 == 0:
            curr_loss = fcn(X, y, beta)
            relative = abs(curr_loss - prev_loss) / (abs(prev_loss) + 1e-8)

            if relative < tol:
                print(f"SGD Converged at iteration {k}")
                break
                
            prev_loss = curr_loss

    return {"value": beta, "iterations": num_iter}

## 3. Stochastic Gradient Descent with AMSGrad Adam (which adjusts step sizes per parameter using momentum and adaptive scaling).

In [121]:
def AMSGrad_Adam_SGD(gradient, X, y, batch_size = 25, step_size = 0.05):
    # Parameters
    tol = 1e-4
    tot_iter = 50000
    beta_1 = 0.9
    beta_2 = 0.99
    
    # Setup
    num_obs = X.shape[0]
    beta = np.zeros(X.shape[1])
    m = np.zeros_like(beta)  # initialize First moment
    z = np.zeros_like(beta)  # initialize Second moment
    prev_loss = fcn(X, y, beta)
    num_iter = 0
    
    for k in range(1, tot_iter + 1):
        # Increment
        num_iter += 1
        
        # SUBSETING OBS RANDOMLY (SGD)
        index = np.random.choice(num_obs, batch_size, replace=False)
        X_sub = X[index]
        y_sub = y[index]

        ### === FULL UPDATE === ###
        
        # COMPUTE GRADIENT
        grad = gradient(X_sub, y_sub, beta)

        # MOMENTS
        m = beta_1 * m + (1 - beta_1) * grad
        z_last = z.copy()
        z = beta_2 * z + (1 - beta_2) * (grad * grad)
        z = np.maximum(z, z_last)
        
        # BIAS CORRECTION
        m_hat = m / (1 - beta_1**k)
        z_hat = z / (1 - beta_2**k)
        
        # COMPUTE NEW POINT
        beta = beta - step_size * (m_hat / (np.sqrt(z_hat) + 1e-8))

        # STOP CRITERION (CHECK EVERY 20 TIMES FOR STABILITY)
        if k % 20 == 0:
            curr_loss = fcn(X, y, beta)
            relative = abs(curr_loss - prev_loss) / (abs(prev_loss) + 1e-8)

            if relative < tol:
                print(f"Adam Converged at iteration {k}")
                break
                
            prev_loss = curr_loss

    return {"value": beta, "iterations": num_iter}

## 4. Stochastic Gradient Descent with AMSGrad AdamW (which incorporates decoupled weight decay alongside momentum and adaptive scaling).

In [123]:
def AMSGrad_AdamW_SGD(gradient, X, y, batch_size = 25, step_size = 0.05):
    # Parameters
    tol = 1e-4
    tot_iter = 50000
    lamb = 1e-4
    beta_1 = 0.9
    beta_2 = 0.999
    
    # Setup
    num_obs = X.shape[0]
    beta = np.zeros(X.shape[1])
    points = [beta.copy()]
    m = np.zeros_like(beta)  # initialize First moment
    z = np.zeros_like(beta)  # initialize Second moment
    prev_loss = fcn(X, y, beta)
    num_iter = 0
    
    for k in range(1, tot_iter + 1):
        # Increment
        num_iter += 1
        
        # SUBSETING OBS RANDOMLY (SGD)
        index = np.random.choice(num_obs, batch_size, replace=False)
        X_sub = X[index]
        y_sub = y[index]

        ### === FULL UPDATE === ###
        
        # COMPUTE GRADIENT
        grad = gradient(X_sub, y_sub, beta)

        # MOMENTS
        m = beta_1 * m + (1 - beta_1) * grad
        z_last = z.copy()
        z = beta_2 * z + (1 - beta_2) * (grad * grad)
        z = np.maximum(z, z_last)
        
        # BIAS CORRECTION
        m_hat = m / (1 - beta_1**k)
        z_hat = z / (1 - beta_2**k)
        
        # COMPUTE NEW POINT
        beta = beta - step_size * (m_hat / (np.sqrt(z_hat) + 1e-8)) - step_size * lamb * beta
        points.append(beta.copy())

        # STOP CRITERION (CHECK EVERY 20 TIMES FOR STABILITY)
        if k % 20 == 0:
            curr_loss = fcn(X, y, beta)
            relative = abs(curr_loss - prev_loss) / (abs(prev_loss) + 1e-8)

            if relative < tol:
                print(f"AdamW Converged at iteration {k}")
                break
                
            prev_loss = curr_loss

    return {"value": beta, "iterations": num_iter}

## Read Dataset

In [17]:
data35 = pd.read_csv("mnist_3_vs_5.csv")
data49 = pd.read_csv("mnist_4_vs_9.csv")

In [ ]:
# 35 - GLM 
data35["label"] = np.where(data35["label"] == 5, 1, 0) 
data35["label"] = data35["label"].astype(float) 
y35 = data35["label"].values 
X35 = data35.drop(columns=["label"]) 
X35 = X35.values 
glm_35 = sm.GLM(y35, X35, family=sm.families.Binomial()).fit() 
# print(glm_35.params)

In [ ]:
# 49 - GLM 
data49["label"] = np.where(data49["label"] == 9, 1, 0) 
data49["label"] = data49["label"].astype(float) 
y49 = data49["label"].values 
X49 = data49.drop(columns=["label"]) 
X49 = X49.values 
glm_49 = sm.GLM(y49, X49, family=sm.families.Binomial()).fit() 
# print(glm_49.params)

# TASK 1: Classifying digit 3 vs. 5

In [43]:
# s = 1 (35 Classification)
s = 1
np.random.seed(10)
# Convergence
Nesterov_35 = Nesterov_SGD(gradient, X35, y35, batch_size = s, step_size = 0.1)
SGD_35 = SGD(gradient, X35, y35, batch_size = 1, step_size_0 = s)
Adam_35 = AMSGrad_Adam_SGD(gradient, X35, y35, batch_size = s, step_size = 0.02)
AdamW_35 = AMSGrad_AdamW_SGD(gradient, X35, y35, batch_size = s, step_size = 0.03)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_35['value'] - glm_35.params)**2}")
beta_Nes_35 = Nesterov_35["value"]
print(f"Estimation error (SGD): { norm2(SGD_35['value'] - glm_35.params)**2}")
beta_SGD_35 = SGD_35["value"]
print(f"Estimation error (Adam): { norm2(Adam_35['value'] - glm_35.params)**2}")
beta_Adam_35 = Adam_35["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_35['value'] - glm_35.params)**2}")
beta_AdamW_35 = AdamW_35["value"]

Nesterov Converged at iteration 1580
SGD Converged at iteration 3320
SGD Converged at iteration 3460
SGD Converged at iteration 2520
Estimation error (Nesterov): 34947010384.13662
Estimation error (SGD): 34945028469.16506
Estimation error (Adam): 34945105565.63969
Estimation error (AdamW): 34945226034.27299


In [51]:
# s = 10 (35 Classification)
s = 10
np.random.seed(10)
# Convergence
Nesterov_35 = Nesterov_SGD(gradient, X35, y35, batch_size = s, step_size = 0.1)
SGD_35 = SGD(gradient, X35, y35, batch_size = 1, step_size_0 = 0.5)
Adam_35 = AMSGrad_Adam_SGD(gradient, X35, y35, batch_size = s, step_size = 0.03)
AdamW_35 = AMSGrad_AdamW_SGD(gradient, X35, y35, batch_size = s, step_size = 0.02)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_35['value'] - glm_35.params)**2}")
beta_Nes_35 = Nesterov_35["value"]
print(f"Estimation error (SGD): { norm2(SGD_35['value'] - glm_35.params)**2}")
beta_SGD_35 = SGD_35["value"]
print(f"Estimation error (Adam): { norm2(Adam_35['value'] - glm_35.params)**2}")
beta_Adam_35 = Adam_35["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_35['value'] - glm_35.params)**2}")
beta_AdamW_35 = AdamW_35["value"]

Nesterov Converged at iteration 9700
SGD Converged at iteration 2140
SGD Converged at iteration 1680
SGD Converged at iteration 1820
Estimation error (Nesterov): 34954912393.396385
Estimation error (SGD): 34945028515.262695
Estimation error (Adam): 34945200996.31968
Estimation error (AdamW): 34945308452.262985


In [55]:
# s = 50 (35 Classification)
s = 50
np.random.seed(10)
# Convergence
Nesterov_35 = Nesterov_SGD(gradient, X35, y35, batch_size = s, step_size = 0.1)
SGD_35 = SGD(gradient, X35, y35, batch_size = 1, step_size_0 = 0.5)
Adam_35 = AMSGrad_Adam_SGD(gradient, X35, y35, batch_size = s, step_size = 0.03)
AdamW_35 = AMSGrad_AdamW_SGD(gradient, X35, y35, batch_size = s, step_size = 0.04)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_35['value'] - glm_35.params)**2}")
beta_Nes_35 = Nesterov_35["value"]
print(f"Estimation error (SGD): { norm2(SGD_35['value'] - glm_35.params)**2}")
beta_SGD_35 = SGD_35["value"]
print(f"Estimation error (Adam): { norm2(Adam_35['value'] - glm_35.params)**2}")
beta_Adam_35 = Adam_35["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_35['value'] - glm_35.params)**2}")
beta_AdamW_35 = AdamW_35["value"]

Nesterov Converged at iteration 6200
SGD Converged at iteration 5260
SGD Converged at iteration 5120
SGD Converged at iteration 4320
Estimation error (Nesterov): 34944359658.92033
Estimation error (SGD): 34945028395.978096
Estimation error (Adam): 34945877314.61677
Estimation error (AdamW): 34946709521.34596


In [60]:
# s = 500 (35 Classification)
s = 500
np.random.seed(10)
# Convergence
Nesterov_35 = Nesterov_SGD(gradient, X35, y35, batch_size = s, step_size = 0.1)
SGD_35 = SGD(gradient, X35, y35, batch_size = 1, step_size_0 = 0.5)
Adam_35 = AMSGrad_Adam_SGD(gradient, X35, y35, batch_size = s, step_size = 0.03)
AdamW_35 = AMSGrad_AdamW_SGD(gradient, X35, y35, batch_size = s, step_size = 0.04)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_35['value'] - glm_35.params)**2}")
beta_Nes_35 = Nesterov_35["value"]
print(f"Estimation error (SGD): { norm2(SGD_35['value'] - glm_35.params)**2}")
beta_SGD_35 = SGD_35["value"]
print(f"Estimation error (Adam): { norm2(Adam_35['value'] - glm_35.params)**2}")
beta_Adam_35 = Adam_35["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_35['value'] - glm_35.params)**2}")
beta_AdamW_35 = AdamW_35["value"]

Nesterov Converged at iteration 1580
SGD Converged at iteration 6860
SGD Converged at iteration 6320
SGD Converged at iteration 5520
Estimation error (Nesterov): 34944961959.19154
Estimation error (SGD): 34945028397.93257
Estimation error (Adam): 34947620754.58543
Estimation error (AdamW): 34948822956.10348


# TASK 2: Classifying digit 4 vs. 9

In [125]:
# s = 1 (49 Classification)
s = 1
np.random.seed(10)
# Convergence
Nesterov_49 = Nesterov_SGD(gradient, X49, y49, batch_size = s, step_size = 0.01)
SGD_49 = SGD(gradient, X49, y49, batch_size = s, step_size_0 = 0.5)
Adam_49 = AMSGrad_Adam_SGD(gradient, X49, y49, batch_size = s, step_size = 0.01)
AdamW_49 = AMSGrad_AdamW_SGD(gradient, X49, y49, batch_size = s, step_size = 0.003)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_49['value'] - glm_49.params)**2}")
beta_Nes_49 = Nesterov_49["value"]
print(f"Estimation error (SGD): { norm2(SGD_49['value'] - glm_49.params)**2}")
beta_SGD_49 = SGD_49["value"]
print(f"Estimation error (Adam): { norm2(Adam_49['value'] - glm_49.params)**2}")
beta_Adam_49 = Adam_49["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_49['value'] - glm_49.params)**2}")
beta_AdamW_49 = AdamW_49["value"]

Nesterov Converged at iteration 4840
SGD Converged at iteration 2760
Adam Converged at iteration 1160
AdamW Converged at iteration 1700
Estimation error (Nesterov): 48576513197.9089
Estimation error (SGD): 48576107544.58708
Estimation error (Adam): 48576108706.091286
Estimation error (AdamW): 48576108042.24727


In [126]:
# s = 10 (49 Classification)
s = 10
np.random.seed(10)
# Convergence
Nesterov_49 = Nesterov_SGD(gradient, X49, y49, batch_size = s, step_size = 0.04)
SGD_49 = SGD(gradient, X49, y49, batch_size = s, step_size_0 = 0.5)
Adam_49 = AMSGrad_Adam_SGD(gradient, X49, y49, batch_size = s, step_size = 0.03)
AdamW_49 = AMSGrad_AdamW_SGD(gradient, X49, y49, batch_size = s, step_size = 0.003)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_49['value'] - glm_49.params)**2}")
beta_Nes_49 = Nesterov_49["value"]
print(f"Estimation error (SGD): { norm2(SGD_49['value'] - glm_49.params)**2}")
beta_SGD_49 = SGD_49["value"]
print(f"Estimation error (Adam): { norm2(Adam_49['value'] - glm_49.params)**2}")
beta_Adam_49 = Adam_49["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_49['value'] - glm_49.params)**2}")
beta_AdamW_49 = AdamW_49["value"]

Nesterov Converged at iteration 4740
SGD Converged at iteration 800
Adam Converged at iteration 1720
AdamW Converged at iteration 2220
Estimation error (Nesterov): 48576224165.167465
Estimation error (SGD): 48576107605.4956
Estimation error (Adam): 48575963104.60183
Estimation error (AdamW): 48576083678.361824


In [128]:
# s = 50 (49 Classification)
s = 50
np.random.seed(10)
# Convergence
Nesterov_49 = Nesterov_SGD(gradient, X49, y49, batch_size = s, step_size = 0.07)
SGD_49 = SGD(gradient, X49, y49, batch_size = s, step_size_0 = 0.5)
Adam_49 = AMSGrad_Adam_SGD(gradient, X49, y49, batch_size = s, step_size = 0.03)
AdamW_49 = AMSGrad_AdamW_SGD(gradient, X49, y49, batch_size = s, step_size = 0.03)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_49['value'] - glm_49.params)**2}")
beta_Nes_49 = Nesterov_49["value"]
print(f"Estimation error (SGD): { norm2(SGD_49['value'] - glm_49.params)**2}")
beta_SGD_49 = SGD_49["value"]
print(f"Estimation error (Adam): { norm2(Adam_49['value'] - glm_49.params)**2}")
beta_Adam_49 = Adam_49["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_49['value'] - glm_49.params)**2}")
beta_AdamW_49 = AdamW_49["value"]

Nesterov Converged at iteration 5360
SGD Converged at iteration 680
Adam Converged at iteration 7060
AdamW Converged at iteration 1440
Estimation error (Nesterov): 48575965866.84659
Estimation error (SGD): 48576107593.2995
Estimation error (Adam): 48575664052.802185
Estimation error (AdamW): 48575468085.253746


In [129]:
# s = 500 (49 Classification)
s = 500
np.random.seed(10)
# Convergence
Nesterov_49 = Nesterov_SGD(gradient, X49, y49, batch_size = s, step_size = 0.1)
SGD_49 = SGD(gradient, X49, y49, batch_size = s, step_size_0 = 0.5)
Adam_49 = AMSGrad_Adam_SGD(gradient, X49, y49, batch_size = s, step_size = 0.03)
AdamW_49 = AMSGrad_AdamW_SGD(gradient, X49, y49, batch_size = s, step_size = 0.03)

# Estimation error
print(f"Estimation error (Nesterov): { norm2(Nesterov_49['value'] - glm_49.params)**2}")
beta_Nes_49 = Nesterov_49["value"]
print(f"Estimation error (SGD): { norm2(SGD_49['value'] - glm_49.params)**2}")
beta_SGD_49 = SGD_49["value"]
print(f"Estimation error (Adam): { norm2(Adam_49['value'] - glm_49.params)**2}")
beta_Adam_49 = Adam_49["value"]
print(f"Estimation error (AdamW): { norm2(AdamW_49['value'] - glm_49.params)**2}")
beta_AdamW_49 = AdamW_49["value"]

Nesterov Converged at iteration 1520
SGD Converged at iteration 540
Adam Converged at iteration 5440
AdamW Converged at iteration 1080
Estimation error (Nesterov): 48576083376.01849
Estimation error (SGD): 48576107595.559784
Estimation error (Adam): 48575512149.17073
Estimation error (AdamW): 48575637634.06924
